In [1]:
# Checking .yaml load/

In [2]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [3]:
from config_handler import initiate_config, load_config

In [4]:
initiate_config()

[INFO] Config file not found at 'config.yaml'.
→ Starting interactive setup.

--- CONFIG SETUP ---
The 'data_path' is mandatory and should be the **FOLDER** where the 'ds004504' directory is located.



data_path []:  /Users/user/eeg-ds004504
derivatives [True]:  
windowLength [3]:  
stepSize [1.5]:  



Define frequency bands of interest.


Use default freqBands? (y/n) [{'Delta': [0.5, 4], 'Theta': [4, 8], 'Alpha': [8, 12], 'Beta': [12, 30]}]:  


[INFO] New config file created at 'config.yaml'


{'data_path': '/Users/user/eeg-ds004504',
 'derivatives': True,
 'windowLength': 3,
 'stepSize': 1.5,
 'freqBands': {'Delta': [0.5, 4],
  'Theta': [4, 8],
  'Alpha': [8, 12],
  'Beta': [12, 30]}}

In [5]:
print(load_config())

{'data_path': '/Users/user/eeg-ds004504', 'derivatives': True, 'windowLength': 3, 'stepSize': 1.5, 'freqBands': {'Delta': [0.5, 4], 'Theta': [4, 8], 'Alpha': [8, 12], 'Beta': [12, 30]}}


# TODO: make it so that everything uses the config file:
redo the 'return path' functions, and rely on config file if not given input :)and rely of n config file if not given input :).
prompted gpt 'april 8' so can check that chat at the bottom I think its pretty good 

In [ ]:
print(load_config())

In [ ]:
# TODO: make functions less verbose for data creation
# TODO: clean import statments
# TODO: see if we are calculating Total Energy correctly, I think the whole row is all 0 after std so ... 
# TODO: Experiment with the derivatives and  not derivatives data (preprocessed and 'raw' respectively, I think currently raw_

In [ ]:
 # this is a good little tutorial to understand basics of pyspark  
# https://domino.ai/blog/principal-component-analysis-pca-on-large-neuroimaging-datasets-using-pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [ ]:
# Spark is a library that distributes the load of computation/ram very efficiently and evenly :)

In [ ]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [ ]:
# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

In [ ]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

In [ ]:
subject_df = load_subjects_df(spark, participants_path="/Users/user/eeg-ds004504/ds004504/participants.tsv") #this is the .tsv with the information of all the participants

In [ ]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import set_data_path, get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
sub1.show()

In [ ]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

In [ ]:
result_group_a.columns

In [ ]:
#Since we doni't want to recreate the data all the time, lets save it and I will see you in Example_Data_Processing

In [ ]:
type(group_a_spark_df)

In [ ]:
group_a_pandas_df = group_a_spark_df.toPandas() # see here, spark has its own data frame type with lots of its own functions
group_c_pandas_df = group_c_spark_df.toPandas() # Each .pkl is around 50mb last time I checked


In [ ]:
group_a_pandas_df.to_pickle("features_alz_example_post_config.pkl") # pkl is a way to store python dataframes, its nice
group_c_pandas_df.to_pickle("features_cntrl_example_post_config.pkl") # pkl is a way to store python dataframes, its nice

In [ ]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_example_post_config.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_example_post_config.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [ ]:
type(group_a_spark_df)

In [ ]:
if group_a_spark_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

In [ ]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)